In [2]:
import os
from pyspark.sql import SparkSession

# Forzamos la detención de cualquier rastro anterior
try:
    spark.stop()
except:
    pass

# Iniciamos una sesión limpia y ligera
spark = SparkSession.builder \
    .appName("Conexion_Directa") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:3.0.1") \
    .config("spark.mongodb.input.uri", "mongodb://mongodb:27017/MercadoLibre.e-commerce") \
    .getOrCreate()

print("✅ ¡Conectado! Ahora intenta cargar el DataFrame.")

# Solo una línea para probar
df = spark.read.format("com.mongodb.spark.sql.DefaultSource").load()
print(f"Total registros: {df.count()}")
df.show(5)

✅ ¡Conectado! Ahora intenta cargar el DataFrame.
Total registros: 143
+--------------------+------------+-------------------+----------+--------------------+---------+-----------+
|                 _id|calificacion|      fecha_captura|     grupo|       identificador|    marca|      valor|
+--------------------+------------+-------------------+----------+--------------------+---------+-----------+
|{69df99a56ceaad0d...|         4.8|2026-04-15 13:58:17|e-commerce|Notebook HP 15-fc...|       HP|5.3499017E7|
|{69df99a56ceaad0d...|         4.9|2026-04-15 13:58:17|e-commerce|Notebook Hp 250 G...|       HP|4.1899022E7|
|{69df99a56ceaad0d...|         4.7|2026-04-15 13:58:17|e-commerce|Notebook HP 14-ep...|       HP|6.2999024E7|
|{69df99a56ceaad0d...|         4.8|2026-04-15 13:58:18|e-commerce|15.6 Notebook Int...|KINGBOLEN|3.5699928E7|
|{69df99a56ceaad0d...|         4.9|2026-04-15 13:58:18|e-commerce|Notebook Hp 15-fc...|       HP|4.1999027E7|
+--------------------+------------+---------------

In [3]:
from pyspark.sql.functions import col, split, when, avg
# 2. Transformación de Negocio: Extraer la MARCA
# En Amazon, la primera palabra del título suele ser la marca.
df_transformado = df.withColumn("marca", split(col("identificador"), " ")[0])

# 3. Limpieza de Outliers: Filtrar laptops con precios erróneos (ej: < 100 euros)
df_validado = df_transformado.filter(col("valor") > 100)

# 4. Agregación: Precio promedio por Marca
reporte_marcas = df_validado.groupBy("marca").agg(
    avg("valor").alias("precio_promedio")
).orderBy(col("precio_promedio").desc())

reporte_marcas.show()

+---------+------------------+
|    marca|   precio_promedio|
+---------+------------------+
|      MSI|            2595.8|
|       LG|            1301.5|
|     DELL|            929.75|
|     Acer|             909.4|
|   Lenovo| 870.3684210526316|
|     ASUS| 846.6153846153846|
|Microsoft|             788.0|
| ACEMAGIC| 510.1111111111111|
|     acer| 505.6666666666667|
|  Lenovo,|             492.5|
|      16"|             439.0|
|    Ryzen|             424.0|
|        2| 412.3333333333333|
|      GMR|             402.0|
|    15.6"|             399.0|
|   Laptop| 371.6923076923077|
|     15.6|             358.5|
|       HP|357.57142857142856|
|     2026|             299.0|
| PINSTONE|             256.0|
+---------+------------------+
only showing top 20 rows

